# Stage 2.5 queue-ownership repair paired evaluation

This notebook compares the existing persistent worker queues with the same candidate configuration plus the new default-off `queue_ownership_repair` flag. The candidate remains stochastic P-final PPO; the opponent is frozen BC-E. Common controls are `underfoot-first`, deadline-safe planting, and deadline-safe hiring. `E_LEGACY`, standard mixed opening, both orientations, and the original episode identity formula are fixed. Schedule-informed hiring is deliberately off.

The first evaluation is the known seed `1470672056`, both orientations, with capture enabled. After checkpoint-free targeted regression and that capture run, the notebook exposes the 32-seed panel. Captured telemetry is reported only when the executor capture or audit actually exposes it; no checkpoint result is prefilled.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, os, shutil, subprocess, sys, tempfile, time

REPO_URL = 'https://github.com/BillXu21/Kaggriculture.git'
EXPERIMENT_REF = 'codex/stage25-upkeep-ablation'
# Pinned evaluator/source commit; the notebook pin is kept one commit behind
# this notebook update so the cloned source contains the evaluator artifacts.
CODE_SHA = '9d0035a'
PPO_CHECKPOINT = Path('/kaggle/working/interactive_curriculum_0acfe858/runs/P_bce_fullspace_lr3e5_from_Ou10_s43049/final.npz')
BC_E_CHECKPOINT = Path('/kaggle/input/datasets/billll/v0-bc-e/best.pt')
KNOWN_SEED = 1470672056
ORIGINAL_16_SEEDS = [144368101, 309507, 615013, 918079, 1221109, 1524137, 1827169, 2130193, 2433221, 2736251, 3039283, 3342311, 3645341, 3948373, 4251401, 2112243121]
# Deterministic extension: affine transform of the first 15 repository-panel seeds,
# followed by the known capture seed. The order is part of episode identity.
PANEL_EXTENSION = [995106988, 1303793286, 521973470, 107449192, 768565387, 1370134739, 2090797777, 425789796, 1027359148, 1688475343, 261654734, 863224086, 1524340281, 97519672, 699089024]
PANEL_SEEDS = ORIGINAL_16_SEEDS + [KNOWN_SEED] + PANEL_EXTENSION
MASTER_SEED = 25
VARIANTS = ['combined', 'combined_wheat3']
PROCESSES = 4
E_HISTORY_VERSION = 'E_LEGACY'
BACKEND = 'official'
SCHEDULE_INFORMED_HIRING = False
COMMON_FLAGS = ['--underfoot-first', '--deadline-safe-planting', '--deadline-safe-hiring']
CONTROL_FLAGS = COMMON_FLAGS + ['--persistent-worker-queues']
TREATMENT_FLAGS = CONTROL_FLAGS + ['--queue-ownership-repair']
if CODE_SHA.startswith('REPLACE_'):
    raise RuntimeError('Edit CODE_SHA to the commit containing this notebook and evaluator before running')
if len(PANEL_SEEDS) != 32 or len(set(PANEL_SEEDS)) != 32 or KNOWN_SEED not in PANEL_SEEDS:
    raise ValueError('PANEL_SEEDS must be a unique ordered 32-seed panel containing KNOWN_SEED')
for path in (PPO_CHECKPOINT, BC_E_CHECKPOINT):
    if not path.is_file():
        raise FileNotFoundError(f'Missing editable checkpoint path: {path}')
RUN_ROOT = Path('/kaggle/working') / ('stage25_queue_repair_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
RUN_ROOT.mkdir(parents=True, exist_ok=False)
REPO = RUN_ROOT / 'repo'
print('Run root:', RUN_ROOT)
print('Ordered 32-seed panel:', PANEL_SEEDS)

In [ ]:
# Authenticate through a short-lived askpass process. The secret is never in the URL, argv, or output.
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Kaggle Secret GITHUB_TOKEN is empty')
try:
    with tempfile.TemporaryDirectory(prefix='stage25_askpass_') as tmp:
        askpass = Path(tmp) / 'askpass.py'
        askpass.write_text("import os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['STAGE25_TOKEN'])\n")
        env = {**os.environ, 'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'STAGE25_TOKEN': token}
        subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', EXPERIMENT_REF, REPO_URL, str(REPO)], env=env, check=True, timeout=180)
        subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CODE_SHA], cwd=REPO, env=env, check=True, timeout=180)
        subprocess.run(['git', 'checkout', '--detach', CODE_SHA], cwd=REPO, env=env, check=True, timeout=30)
finally:
    token = None
    if 'env' in globals():
        env.pop('STAGE25_TOKEN', None)
actual_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
if actual_sha != CODE_SHA:
    raise RuntimeError(f'checkout mismatch: {actual_sha} != {CODE_SHA}')
print('Pinned source:', actual_sha)

In [ ]:
# Preserve the existing bounded one-CPU-per-child setup.
EVAL_ENV = os.environ.copy()
EVAL_ENV.update({
    'PYTHONUNBUFFERED': '1', 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1',
    'OPENBLAS_NUM_THREADS': '1', 'NUMEXPR_NUM_THREADS': '1',
    'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
    'XLA_FLAGS': '--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1',
    'JAX_PLATFORMS': 'cpu', 'PYTHONPATH': str(REPO),
})
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
tracked = [REPO / 'executor_v0/agent.py', REPO / 'executor_v0/foreman.py', REPO / 'executor_v0/scheduler.py', REPO / 'tools/evaluate_stage25_upkeep.py', REPO / 'tools/run_stage25_upkeep_sharded.py']
source_hashes = {str(path.relative_to(REPO)): sha256(path) for path in tracked}
print(json.dumps({'source_commit': actual_sha, 'source_hashes': source_hashes, 'ppo_sha256': sha256(PPO_CHECKPOINT), 'bc_e_sha256': sha256(BC_E_CHECKPOINT)}, indent=2))

def preflight(name, seeds, flags):
    output = RUN_ROOT / (name + '_preflight')
    command = [sys.executable, '-m', 'tools.run_stage25_upkeep_sharded', '--preflight-only', '--output-dir', str(output), '--backend', BACKEND, '--e-history-version', E_HISTORY_VERSION, '--master-seed', str(MASTER_SEED), '--processes', str(PROCESSES), '--seeds', *map(str, seeds), '--variants', *VARIANTS, *flags]
    subprocess.run(command, cwd=REPO, env=EVAL_ENV, check=True)
    payload = json.loads((output / 'preflight.json').read_text())
    assert payload['ordered_seeds'] == list(seeds)
    return payload

# Check wrapper identity and the exact model-loading imports before the known seed.
preflight('known_seed', [KNOWN_SEED], CONTROL_FLAGS)
preflight('panel', PANEL_SEEDS, TREATMENT_FLAGS)
subprocess.run([sys.executable, '-c', 'import bc_manager_jax, rl_manager, kaggle_environments'], cwd=REPO, env=EVAL_ENV, check=True)
print('Preflight and model-loading imports passed; no evaluation games have run yet.')

In [ ]:
def run_arm(name, seeds, extra_flags, capture):
    output = RUN_ROOT / name
    command = [sys.executable, '-m', 'tools.run_stage25_upkeep_sharded', '--checkpoint', str(PPO_CHECKPOINT), '--e-checkpoint', str(BC_E_CHECKPOINT), '--output-dir', str(output), '--backend', BACKEND, '--e-history-version', E_HISTORY_VERSION, '--master-seed', str(MASTER_SEED), '--processes', str(PROCESSES), '--seeds', *map(str, seeds), '--variants', *VARIANTS, *extra_flags]
    capture_dir = RUN_ROOT / (name + '_captures') if capture else None
    if capture_dir is not None:
        command += ['--capture-dir', str(capture_dir)]
    (RUN_ROOT / (name + '.command.json')).write_text(json.dumps({'command': command, 'source_commit': actual_sha, 'capture_neutrality': 'capture is passive; actions are computed before snapshots', 'schedule_informed_hiring': SCHEDULE_INFORMED_HIRING}, indent=2))
    print(f'Starting {name}: {len(seeds) * 2 * len(VARIANTS)} evaluator rows', flush=True)
    started = time.monotonic()
    process = subprocess.Popen(command, cwd=REPO, env=EVAL_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    log_path = RUN_ROOT / (name + '.log')
    with log_path.open('w') as log:
        for line in process.stdout:
            log.write(line); log.flush(); print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'{name} failed with {code}; partial outputs remain at {output}')
    print(f'{name} completed in {(time.monotonic() - started) / 60:.1f} minutes', flush=True)
    return output, capture_dir

if SCHEDULE_INFORMED_HIRING:
    raise AssertionError('schedule-informed hiring must remain off')
# Known capture first: both orientations are retained by the full seed list.
known_control, known_control_capture = run_arm('known_seed_persistent_queues', [KNOWN_SEED], CONTROL_FLAGS, capture=True)
known_treatment, known_treatment_capture = run_arm('known_seed_persistent_queues_repair', [KNOWN_SEED], TREATMENT_FLAGS, capture=True)
subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_stage25_sharded.py', 'tests/test_stage25_capture.py', '-q'], cwd=REPO, env=EVAL_ENV, check=True)
print('Known-seed capture and targeted regression passed; proceeding to the full panel.')
# Only after the known capture and targeted regression does the full 32-seed panel run.
panel_control, _ = run_arm('panel_persistent_queues', PANEL_SEEDS, CONTROL_FLAGS, capture=False)
panel_treatment, _ = run_arm('panel_persistent_queues_repair', PANEL_SEEDS, TREATMENT_FLAGS, capture=False)

In [ ]:
SURFACE_FIELDS = ['candidate_bank', 'opponent_bank', 'margin', 'hiring_expense', 'completed_useful_work', 'missed_maintenance', 'duplicate_claims', 'target_abandonment', 'movement_between_interactions']
def load_rows(output):
    manifest = json.loads((output / 'manifest.json').read_text())
    rows = [json.loads(line) for line in (output / 'games.jsonl').read_text().splitlines() if line.strip()]
    expected = len(manifest['ordered_seeds']) * 2 * len(manifest['variants'])
    if manifest.get('status') != 'complete' or manifest['ordered_seeds'] != (PANEL_SEEDS if 'panel' in output.name else [KNOWN_SEED]) or len(rows) != expected:
        raise RuntimeError(f'incomplete or reordered output: {output}')
    return manifest, rows, json.loads((output / 'telemetry.json').read_text())

def unavailable(reason):
    return {'available': False, 'reason': reason}

def audit_capture(name, capture_dir):
    audit_dir = RUN_ROOT / (name + '_audit')
    subprocess.run([sys.executable, '-m', 'tools.audit_stage25_capture', '--capture-dir', str(capture_dir), '--output-dir', str(audit_dir), '--baseline', 'combined', '--focus-pairs', '2'], cwd=REPO, env=EVAL_ENV, check=True)
    return audit_dir

def audit_surface(audit_dir):
    path = audit_dir / 'games.csv'
    if not path.is_file():
        return {}
    rows = list(csv.DictReader(path.open(newline='', encoding='utf-8')))
    def json_value(row, key):
        try: return json.loads(row.get(key, ''))
        except (TypeError, ValueError, json.JSONDecodeError): return None
    completed = sum(sum((json_value(row, 'cand_completed') or {}).values()) for row in rows)
    abandoned = sum(sum((json_value(row, 'cand_ended_unobserved') or {}).values()) for row in rows)
    return {
        'completed_useful_work': {'available': bool(rows), 'value': completed, 'source': 'capture audit observed-interaction heuristic'},
        'duplicate_claims': {'available': bool(rows), 'value': sum(float(row.get('cand_coassigned_turns') or 0) for row in rows), 'source': 'capture audit coassigned_turns'},
        'target_abandonment': {'available': bool(rows), 'value': abandoned, 'source': 'capture audit ended_unobserved'},
        'movement_between_interactions': {'available': bool(rows), 'value': sum(float(row.get('cand_movement') or 0) for row in rows), 'source': 'capture audit cand_movement worker-turn proxy'},
    }

def report(name, output, audit_dir=None):
    manifest, rows, telemetry = load_rows(output)
    audit_fields = audit_surface(audit_dir) if audit_dir else {}
    surface = {}
    for field in SURFACE_FIELDS:
        surface[field] = audit_fields.get(field) or telemetry.get('fields', {}).get(field) or unavailable('capture/audit did not expose this field')
    payload = {'arm': name, 'source_commit': manifest.get('source', {}).get('commit'), 'games': len(rows), 'surface': surface}
    print(json.dumps(payload, indent=2))
    return payload

known_control_audit = audit_capture('known_seed_persistent_queues', known_control_capture)
known_treatment_audit = audit_capture('known_seed_persistent_queues_repair', known_treatment_capture)
surface_report = {
    'known_control': report('known_control', known_control, known_control_audit),
    'known_treatment': report('known_treatment', known_treatment, known_treatment_audit),
    'panel_control': report('panel_control', panel_control),
    'panel_treatment': report('panel_treatment', panel_treatment),
}
(RUN_ROOT / 'surface_metrics.json').write_text(json.dumps(surface_report, indent=2) + '\n')
print('No result is promoted here; inspect paired margins and all requested telemetry together.')